In [1]:
import pandas as pd

from helpers import sql

# pandas formatting
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('max_colwidth', 1000)
pd.set_option('display.float_format', '{:.0f}'.format)


In [2]:
df = sql(f"""
    SELECT * FROM
        csas2_document
            JOIN csas2_documenttracking ON csas2_documenttracking.document_id = csas2_document.id
    WHERE csas2_document.id IN (
        1348, 1359, 1349, 1373, 1302, 1352, 1378, 1419, 1379, 1398, 1377, 1416, 1376, 1425, 1380, 1424,
        1445, 1446, 1444, 1415, 1417, 1382, 1303, 1426, 1397, 1466, 1407, 1544, 1301, 1458, 1414, 1429,
        1536, 1537, 1422, 1432, 1451, 1498, 1493, 1494, 1551, 1441, 1442
    )
""")

df.target_lang.value_counts()

target_lang
2    30
1     5
Name: count, dtype: int64

In [3]:
# compare fr and en - are they similar enough to choose 4 random en and all fr docs? (yes, look fine to me)
pd.concat([
    df[df.target_lang==1].describe().T[['mean', 'std']].rename({'mean': 'fr_mean', 'std': 'fr_std'}, axis=1),
    df[df.target_lang==2].describe().T[['mean', 'std']].rename({'mean': 'en_mean', 'std': 'en_std'}, axis=1),
], axis=1).head(20)

,fr_mean,fr_std,en_mean,en_std
id,1432,36,1418,72
created_at,2025-03-07 04:18:20.667719936,NaN,2025-01-30 20:19:06.663366144,NaN
updated_at,2025-07-05 11:06:03.445468672,NaN,2025-08-19 06:36:40.242040064,NaN
pages_en,14,8,17,12
status,12,0,12,0
created_by_id,927,263,1687,1365
process_id,805,13,743,116
updated_by_id,2359,0,2372,48
document_type_id,3,2,3,2
translation_status,3,0,3,1


In [4]:
df_fr = df[df.target_lang==1].copy()
df_fr['lang'] = 'fr'
df_en = df[(df.target_lang==2) & (~df.library_link_en.isnull())].copy().sample(5, random_state=42)
df_en['lang'] = 'en'
columns = ['id', 'lang', 'library_link_en', 'library_link_fr', 'sharepoint_archive_en', 'sharepoint_archive_fr']

df_docs = pd.concat([
    df_fr,
    df_en,
], axis=0)[columns]


In [5]:
# new docs added to list, pick a new seed so that I don't need change already downloaded random docs

old_list = [1301, 1303, 1407, 1432, 1415, 1444, 1458, 1466]
new_list = df_docs.id.to_list()

i = 0
while not all(item in set(new_list) for item in old_list):
    i += 1
    if i == 10000:
        break

    df_en = df[(df.target_lang==2) & (~df.library_link_en.isnull())].copy().sample(5, random_state=i)
    df_en['lang'] = 'en'

    df_docs = pd.concat([
        df_fr,
        df_en,
    ], axis=0)[columns]
    new_list = df_docs.id.to_list()

print(i)

5276


In [6]:
df_en = df[(df.target_lang==2) & (~df.library_link_en.isnull())].copy().sample(5, random_state=5276)
df_en['lang'] = 'en'

df_docs = pd.concat([
    df_fr,
    df_en,
], axis=0)[columns]

In [7]:
df_docs[['id', 'lang', 'sharepoint_archive_en', 'sharepoint_archive_fr']]

,id,lang,sharepoint_archive_en,sharepoint_archive_fr
11,1379,fr,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/MAR/SRR-RS2025-009%20MAR%20English.docx?d=wba92fec6507a46df9bcf9ace9385b9e1&csf=1&web=1&e=Kjg8hj,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/MAR/SRR-RS2025-009%20MAR%20French.docx?d=wb8fbbe2f9bfa48e4bd2c4c32eaca024c&csf=1&web=1&e=yQSkGP
18,1415,fr,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-017%20QC%20English.docx?d=w4bd0b55a41f042a79c358b7118824802&csf=1&web=1&e=21OanB,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-017%20QC%20French.docx?d=w9df845801b5440239498572779b82d68&csf=1&web=1&e=TLF6Rx
30,1444,fr,https://086gc.sharepoint.com/sites/CSASWebandPubTeam/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FCSASWebandPubTeam%2FShared%20Documents%2FDocument%20Archive&clickparams=eyAiWC1BcHBOYW1lIiA6ICJNaWNyb3NvZnQgT3V0bG9vayIsICJYLUFwcFZlcnNpb24iIDogIjE2LjAuMTczMjguMjA3NzAiLCAiT1MiIDogIldpbmRvd3MiIH0%3D,https://086gc.sharepoint.com/sites/CSASWebandPubTeam/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FCSASWebandPubTeam%2FShared%20Documents%2FDocument%20Archive&clickparams=eyAiWC1BcHBOYW1lIiA6ICJNaWNyb3NvZnQgT3V0bG9vayIsICJYLUFwcFZlcnNpb24iIDogIjE2LjAuMTczMjguMjA3NzAiLCAiT1MiIDogIldpbmRvd3MiIH0%3D
34,1458,fr,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-029%20QC%20English.docx?d=w88a1458d0eac4639a485ef5e3f6b4177&csf=1&web=1&e=ucgNiJ,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-029%20QC%20French.docx?d=w2652e067f28043ba9197d6f346f8fde2&csf=1&web=1&e=kzeAgL
35,1466,fr,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-024%20QC%20English.docx?d=w9b6a77c37fe54c3f8874ea84746529bd&csf=1&web=1&e=TBj5em,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/QC/SAR-AS2025-024%20QC%20French.docx?d=wc9070bd957904eeb8c1289d9f5ae1b06&csf=1&web=1&e=1KMTH7
26,1429,en,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/NL/SAR-AS2025-034%20NL%20English.docx?d=w6a6d12b5887c45dea63ec5906c395798&csf=1&web=1&e=kyR29y,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/NL/SAR-AS2025-034%20NL%20French.docx?d=w0e05ad3880844c85bc657e10859a0cd6&csf=1&web=1&e=mgmKaT
16,1407,en,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/PAC/SAR-AS2025-026%20PAC%20English.docx?d=w0ce557d77528468eb7e0c867bfbc97fa&csf=1&web=1&e=rII0Mr,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/PAC/SAR-AS2025-026%20PAC%20French.docx?d=w61068a42337c418ca5032af26cbbba27&csf=1&web=1&e=aGaYoP
0,1301,en,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/PAC/SRR-RS2025-027%20English.docx?d=w08928ac2c89841a1a5ec26f1e2aab7ef&csf=1&web=1&e=0sT6Rf,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/PAC/SRR-RS2025-027%20French.docx?d=w0bb452e98f014978b5aa1f203546741f&csf=1&web=1&e=v6y3Pr
27,1432,en,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/O%26P/SAR-AS2025-042%20ARC%20English.docx?d=w5c973cc2575e43148bb937904122f8c0&csf=1&web=1&e=wOYte6,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/O%26P/SAR-AS2025-042%20ARC%20French.docx?d=w2f482daad7504fa1a73b63076105cee0&csf=1&web=1&e=i29F84
2,1303,en,https://086gc.sharepoint.com/:w:/r/sites/CSASWebandPubTeam/Shared%20Documents/Document%20Archive/PAC/SRR-RS2025-021%20PAC%20English.docx?d=we0d8d8ed38454a01a

In [9]:
# newly added docs to FSAR list (to download manually)
[x for x in new_list if x not in old_list]

[1379, 1429]